# HealthGait — Quick-Start Guide

This notebook covers the full usage cycle:
1. Load the pretrained model from a local checkpoint
2. Inspect preprocessing and model configuration
3. Reconstruct and load the model (architecture auto-detected from the checkpoint)
4. Extract embeddings from skeleton sequences (synthetic)
5. Run inference on a **real sample skeleton** — reads `.csv` **and** `.ntds` clips
6. Simulate **continuing training** (optimizer + scheduler + loss loop)
7. Run an **evaluation pass** (reconstruction loss)
8. **Visualize** true vs predicted skeleton motion as a GIF (README-style cartoon) for every sample

**Model**: `epoch_31.pth`, loaded from a local copy.
**Input**: skeleton sequences of shape `[batch, frames, 26 joints, 4 channels (x, y, z, confidence)]`

> Sections 4, 6 and 7 use **synthetic data** to demonstrate the API. Sections 5 and 8 run real clips through the actual preprocessing.
>
> **Data note:** `.ntds` clips are real subject-motion captures (folders use anonymized IDs). The raw recordings and generated GIFs are git-ignored — do not commit them (see `CLAUDE.md`).

## 0  Setup

In [1]:
# Uncomment if any package is missing
# !pip install torch

In [2]:
import ast, json, sys, os

# Resolve project root — works whether the Jupyter CWD is model/ or the repo root
_cwd = os.getcwd()
if os.path.basename(_cwd) == "model":
    project_root = os.path.dirname(_cwd)
elif os.path.isdir(os.path.join(_cwd, "model", "architecture")):
    project_root = _cwd
else:
    project_root = os.path.abspath(os.path.join(_cwd, ".."))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")

import torch
import numpy as np

from model.architecture.motionBert_full import DSTformer, ReconstructNet
from model.preprocessing.args import PreprocessingArgs
from model.training.utils.training_helper import get_scheduler, load_checkpoint
from model.training.utils.loss import loss_mpjpe, loss_velocity

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Device: {device}")

Project root: /Users/arotem/Documents/GitHub/GaitPredict
Device: mps


Device: mps


## 1  Load Checkpoint (Local)

Pointing at a local copy of the published checkpoint + configs instead of pulling from HuggingFace.

In [3]:
MODEL_DIR = "/Users/arotem/Documents/Models/MotionBertAdam"

checkpoint_path  = os.path.join(MODEL_DIR, "epoch_31.pth")
prep_args_path   = os.path.join(MODEL_DIR, "preprocessing_args.json")
model_args_path  = os.path.join(MODEL_DIR, "model_args.txt")

for p in (checkpoint_path, prep_args_path, model_args_path):
    assert os.path.exists(p), f"Missing file: {p}"

print(f"Checkpoint       : {checkpoint_path}")
print(f"Preprocessing cfg: {prep_args_path}")
print(f"Model args       : {model_args_path}")

Checkpoint       : /Users/arotem/Documents/Models/MotionBertAdam/epoch_31.pth
Preprocessing cfg: /Users/arotem/Documents/Models/MotionBertAdam/preprocessing_args.json
Model args       : /Users/arotem/Documents/Models/MotionBertAdam/model_args.txt


## 2  Inspect Configuration

In [4]:
# ── Preprocessing args ────────────────────────────────────────────────────────
with open(prep_args_path) as f:
    prep_data = json.load(f)
prep_args = PreprocessingArgs(**prep_data)

print("Preprocessing configuration")
print(f"  Activities : {list(prep_args.videos_lens.keys())}")
print(f"  Excluded   : {prep_args.exclude_list}")
print(f"  Joints     : {prep_args.get_num_joints()}  (remove_noise_joints={prep_args.remove_noise_joints})")
print(f"  Augments on: {[k for k, v in prep_args.augments.items() if v]}")
print(f"  Group mask : {prep_args.group_masking}  span={prep_args.span_masking}")
print(f"  Centroid   : {prep_args.centroid_type}")

Preprocessing configuration
  Activities : ['apose', 'romberg_closed', 'romberg_open', 'self_selected_gait_speed', 'sit_to_stand', 'stationary_walk', 'tm_3kmh']
  Excluded   : ['apose', 'romberg_open']
  Joints     : 26  (remove_noise_joints=True)
  Augments on: ['None', 'jittering']
  Group mask : 4  span=16
  Centroid   : second


In [5]:
# ── Model (training) args ─────────────────────────────────────────────────────
with open(model_args_path) as f:
    model_args = ast.literal_eval(f.read())

print("Model / training configuration")
for k, v in model_args.items():
    print(f"  {k}: {v}")

Model / training configuration
  model: OptimizedModule
  dim_in: 4
  dim_out: 3
  dropout_ratio: 0.1
  batch_size: 8
  learning_rate: 0.0008
  adamw_weight_decay: 0.01
  scheduler_type: Annealing
  num_epochs: 32
  warmup_epochs: 0.3
  warmup_ratio: None
  size_seq: 900
  num_joints: 26
  lambda_scale: 0
  lambda_3d_velocity: 0.1
  mask_loss: False
  with_noise: True
  with_rope: True
  dim_representation: 32
  num_heads: 8


## 3  Build and Load Model

Architecture parameters (depth, `dim_feat`, RoPE, sink tokens …) are **auto-detected**
from the checkpoint's weight shapes — no manual config required.

In [6]:
# ── Load raw checkpoint ───────────────────────────────────────────────────────
raw_ckpt = torch.load(checkpoint_path, map_location="cpu")
state    = raw_ckpt["model"]

# Strip torch.compile and DataParallel prefixes if present
if any(k.startswith("_orig_mod.") for k in state):
    state = {k[len("_orig_mod."):]: v for k, v in state.items()}
if any(k.startswith("module.") for k in state):
    state = {k[len("module."):]: v for k, v in state.items()}

print(f"Checkpoint epoch: {raw_ckpt['epoch']}")
print(f"State dict keys (sample): {list(state.keys())[:6]}")

Checkpoint epoch: 31
State dict keys (sample): ['mask_embedding', 'model_backbone.pos_embed', 'model_backbone.blocks_st.0.norm1_s.weight', 'model_backbone.blocks_st.0.norm1_s.bias', 'model_backbone.blocks_st.0.norm1_t.weight', 'model_backbone.blocks_st.0.norm1_t.bias']


In [7]:
# ── Detect architecture from weight shapes ────────────────────────────────────
def detect_arch(sd):
    dim_feat       = sd["model_backbone.pos_embed"].shape[-1]
    num_joints     = sd["model_backbone.pos_embed"].shape[1]
    depth          = sum(1 for k in sd
                         if "model_backbone.blocks_st." in k
                         and k.endswith(".norm1_s.weight"))
    if "model_backbone.pre_logits.fc.weight" in sd:
        dim_rep    = sd["model_backbone.pre_logits.fc.weight"].shape[0]
    else:
        dim_rep    = dim_feat   # Identity pre_logits
    # RoPE: when enabled, temp_embed is a None buffer and is absent from state_dict
    with_rope      = "model_backbone.temp_embed" not in sd
    num_sink       = (sd["model_backbone.sink_tokens"].shape[1]
                      if "model_backbone.sink_tokens" in sd else 0)
    # Linear weight shape is [out_features, in_features]
    dim_in         = sd["joint_embd.weight"].shape[1]
    dim_out        = sd["reconstruct_head.weight"].shape[0]
    # Koleo head is optional — present only if the checkpoint was trained with it
    koleo_dim      = sd["koleo_head.weight"].shape[0] if "koleo_head.weight" in sd else None
    return dict(dim_feat=dim_feat, num_joints=num_joints, depth=depth,
                dim_rep=dim_rep, with_rope=with_rope, num_sink_tokens=num_sink,
                dim_in=dim_in, dim_out=dim_out, koleo_dim=koleo_dim)

arch = detect_arch(state)
print("Detected architecture:")
for k, v in arch.items():
    print(f"  {k}: {v}")

Detected architecture:
  dim_feat: 128
  num_joints: 26
  depth: 8
  dim_rep: 32
  with_rope: True
  num_sink_tokens: 0
  dim_in: 4
  dim_out: 3
  koleo_dim: 128


In [8]:
# ── Instantiate backbone + wrapper ────────────────────────────────────────────
backbone = DSTformer(
    num_joints      = arch["num_joints"],
    dim_in          = arch["dim_in"],
    maxlen          = model_args["size_seq"],
    depth           = arch["depth"],
    drop_rate       = model_args["dropout_ratio"],
    use_rope        = arch["with_rope"],
    use_flash_attn  = False,   # disable for portability; weights are unaffected
    dim_feat        = arch["dim_feat"],
    dim_rep         = arch["dim_rep"],
    num_heads       = model_args["num_heads"],
    num_sink_tokens = arch["num_sink_tokens"],
)

model = ReconstructNet(
    backbone,
    dim_in    = arch["dim_in"],
    dim_out   = arch["dim_out"],
    koleo_dim = arch["koleo_dim"],
)

model.load_state_dict(state)
model = model.to(device).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded — {n_params / 1e6:.2f}M parameters")

Model loaded — 4.35M parameters


## 4  Embedding Extraction (Inference)

Input skeleton sequences have shape `[B, T, J, C]`:
- `B` — batch size
- `T` — frames (model was trained with `T=900`; shorter windows work too)
- `J` — 26 joints
- `C` — 4 channels: `(x, y, z, confidence)`

The model returns:
- `reconstructed` `[B, T, J, 3]` — predicted 3-D joint positions
- `bundle['pooled_pre_logits_embeddings']` `[B, dim_rep × 3]` — subject-level embedding (mean + max + std pooled)

In [9]:
# Synthetic batch — replace with your real skeleton tensors
DEMO_T = 64   # short window for fast demo; use model_args['size_seq'] (900) for full-length
B = 4

x_demo = torch.randn(B, DEMO_T, arch["num_joints"], arch["dim_in"], device=device)

with torch.no_grad():
    reconstructed, bundle = model(x_demo)

embeddings = bundle["pooled_pre_logits_embeddings"]   # [B, dim_rep * 3]

print(f"Input             : {tuple(x_demo.shape)}")
print(f"Reconstructed XYZ : {tuple(reconstructed.shape)}")
print(f"Subject embedding : {tuple(embeddings.shape)}  ← use this for downstream tasks")

Input             : (4, 64, 26, 4)
Reconstructed XYZ : (4, 64, 26, 3)
Subject embedding : (4, 96)  ← use this for downstream tasks


## 5  Inference on a sample skeleton (README demo)

Loads a real clip through the **actual eval-mode preprocessing** — noise-joint removal
(32 → 26), pelvis-centering, per-frame scale normalization — then a forward pass on the
**pristine checkpoint** (this section runs before the training-mutation demo below).

The helpers defined here handle **both** formats and are reused by the visualization section:
- `.csv`  — the anonymized clip shipped in `sample_data/`
- `.ntds` — Azure-Kinect capture sessions (Arrow/Feather; needs `pyarrow`). These carry an
  extra `body_id` column (multi-body tracking) that the loader strips so the layout matches the CSV.

Unlike the synthetic sections, the reconstruction MPJPE here is meaningful: it measures how
well the model reproduces a real skeleton it has never seen in this notebook.

In [10]:
import warnings
import pandas as pd
from model.preprocessing.preprocessing import DualCameraDataset
from model.preprocessing.normalizing_time_and_space import normalize_cycle_and_space

# pyarrow's read_feather emits a benign deprecation FutureWarning — quiet it once.
warnings.filterwarnings("ignore", message=".*read_feather is deprecated.*", category=FutureWarning)

# ── Generic loader + preprocessing (reused by the visualization section) ──────
prep_args.use_memmap  = False
prep_args.load_to_ram = False

class _PrepShim:
    """Minimal stand-in exposing just what the preprocessing methods read, so we can
    reuse the dataset's exact transforms without its cluster-only file discovery
    (which assumes 'front'-named CSVs and known activity folders)."""
    def __init__(self, args, per_joint_amount):
        self.args = args
        self.per_joint_amount = per_joint_amount
        self.centroid = None

_shim = _PrepShim(prep_args, 4 if prep_args.use_confidence else 3)

def load_skeleton_df(path):
    """Load a skeleton clip from .csv or .ntds (Arrow/Feather) — same k4abt schema."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        df = pd.read_csv(path, dtype="float32")
    elif ext == ".ntds":
        df = pd.read_feather(path)                 # needs pyarrow
    else:
        raise ValueError(f"Unsupported skeleton format: {ext}")
    # NTDS carries a multi-body-tracking 'body_id' column absent from the CSV;
    # keep the primary tracked body and drop it so the column layout matches the CSV.
    if "body_id" in df.columns:
        keep = df["body_id"].value_counts().idxmax()
        df = df[df["body_id"] == keep].reset_index(drop=True).drop(columns=["body_id"])
    return df

def preprocess_sample(path, activity="self_selected_gait_speed"):
    """CSV/NTDS → model input [1, T, 26, 4] via the real eval-mode transforms. Returns (x, n_raw)."""
    df        = load_skeleton_df(path)
    skel_raw  = DualCameraDataset._get_np_from_df(_shim, df)                     # [T, 32, 4]
    skel_norm = DualCameraDataset.normalize_seq_fast(_shim, skel_raw, front=True)  # [T, 26, 4]
    np_coords = normalize_cycle_and_space(skel_norm, activity=activity,
                                          normalize_space=prep_args.normalize_space,
                                          normalize_time=prep_args.normalize_time,
                                          num_frames=prep_args.cycle_len)          # pass-through (both flags off)
    T = model_args["size_seq"]
    n = np_coords.shape[0]
    seq = np_coords[:T] if n >= T else np.pad(np_coords, ((0, T - n), (0, 0), (0, 0)), mode="edge")
    x = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)            # [1, T, 26, 4]
    return x, n

# Walk-through example: the shipped anonymized CSV clip
SAMPLE_CSV = os.path.join(project_root, "sample_data",
                          "self_selected_gait_speed", "front__2024-01-09.csv")
x_real, n_raw = preprocess_sample(SAMPLE_CSV, activity="self_selected_gait_speed")

print(f"Sample clip  : {os.path.relpath(SAMPLE_CSV, project_root)}")
print(f"Raw frames   : {n_raw}")
print(f"Model input  : {tuple(x_real.shape)}  (batch, frames, joints, channels)")
print(f"Value range  : [{x_real.min():.3f}, {x_real.max():.3f}]")

Sample clip  : sample_data/self_selected_gait_speed/front__2024-01-09.csv
Raw frames   : 1829
Model input  : (1, 900, 26, 4)  (batch, frames, joints, channels)
Value range  : [-0.793, 1.000]


In [11]:
# ── Forward pass on the real skeleton (pristine checkpoint) ───────────────────
model.eval()
with torch.no_grad():
    reconstructed_real, bundle_real = model(x_real)

embedding_real = bundle_real["pooled_pre_logits_embeddings"]        # [1, 96]
mpjpe_real = loss_mpjpe(reconstructed_real[..., :3], x_real[..., :3]).item()

print(f"Reconstructed XYZ : {tuple(reconstructed_real.shape)}")
print(f"Subject embedding : {tuple(embedding_real.shape)}")
print(f"Reconstruction MPJPE (vs input) : {mpjpe_real:.4f}"
      "   ← low = model reproduces the real skeleton")
print(f"Embedding preview : {embedding_real[0, :8].cpu().numpy().round(3)}")

Reconstructed XYZ : (1, 900, 26, 3)
Subject embedding : (1, 96)
Reconstruction MPJPE (vs input) : 0.0072   ← low = model reproduces the real skeleton
Embedding preview : [-0.412  0.485 -0.888  0.    -0.723 -0.696  0.001 -0.747]


## 6  Continue Training

This mirrors the flow in `model/training/continue_training.py` and `train_loop.py`.

- **Optimizer**: AdamW with the same LR and weight-decay as the original run
- **Scheduler**: cosine-annealing with a short warmup (via `get_scheduler`)
- **Loss**: MPJPE reconstruction + optional velocity term

Uncomment the `load_checkpoint` call to also restore the optimizer and scheduler states
for a true resumption (no learning-rate reset).

In [12]:
# ── Optimizer & scheduler ─────────────────────────────────────────────────────
model.train()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr           = model_args["learning_rate"],
    weight_decay = model_args["adamw_weight_decay"],
    fused        = (device.type == "cuda"),
)

STEPS_PER_EPOCH = 200   # set to len(train_loader) when using real data

scheduler = get_scheduler(optimizer, {
    "scheduler_type"  : model_args["scheduler_type"],
    "num_epochs"      : model_args["num_epochs"],
    "warmup_epochs"   : model_args["warmup_epochs"],
    "warmup_ratio"    : model_args["warmup_ratio"],
    "steps_per_epoch" : STEPS_PER_EPOCH,
    "learning_rate"   : model_args["learning_rate"],
})

# Optional — restore optimizer + scheduler state for exact resumption:
# model, optimizer, scheduler, scaler, start_epoch = load_checkpoint(
#     model, optimizer, scheduler, checkpoint_path, device
# )

start_epoch = raw_ckpt["epoch"] + 1
print(f"Ready. Resuming from epoch {start_epoch} / {model_args['num_epochs']}")
print(f"LR = {optimizer.param_groups[0]['lr']:.2e}")

Ready. Resuming from epoch 32 / 32
LR = 8.00e-04


In [13]:
# ── Simulated training steps ──────────────────────────────────────────────────
# Replace the synthetic tensors below with batches from your DataLoader:
#   for batch in train_loader:
#       x      = batch['data'].to(device)             # [B, T, J, C]
#       target = batch['original'][..., :3].to(device) # [B, T, J, 3]  (XYZ)

lambda_vel = model_args["lambda_3d_velocity"]   # 0.1

DEMO_BATCH = 2

for step in range(1, 6):
    # ── synthetic data (swap with real loader) ────────────────────────────────
    x      = torch.randn(DEMO_BATCH, DEMO_T, arch["num_joints"], arch["dim_in"],  device=device)
    target = torch.randn(DEMO_BATCH, DEMO_T, arch["num_joints"], 3,               device=device)
    # ─────────────────────────────────────────────────────────────────────────

    optimizer.zero_grad()
    reconstructed, _ = model(x)
    pred_xyz = reconstructed[..., :3]

    loss = loss_mpjpe(pred_xyz, target)
    if lambda_vel > 0:
        loss = loss + lambda_vel * loss_velocity(pred_xyz, target)

    loss.backward()
    optimizer.step()
    scheduler.step()

    print(f"  step {step}  loss={loss.item():.4f}  lr={scheduler.get_last_lr()[0]:.2e}")

  step 1  loss=1.8938  lr=8.00e-04
  step 2  loss=1.8384  lr=8.00e-04
  step 3  loss=1.8244  lr=8.00e-04
  step 4  loss=1.8109  lr=8.00e-04
  step 5  loss=1.8498  lr=8.00e-04


  step 3  loss=1.8478  lr=8.00e-04
  step 4  loss=1.7900  lr=8.00e-04


  step 5  loss=1.8291  lr=8.00e-04


## 7  Evaluation Pass

Mirrors `_collect_model_outputs` in `training_helper.py`.  
Reports mean per-joint position error (MPJPE) in the same units as the input coordinates.

In [14]:
model.eval()

eval_losses  = []
all_embeddings = []

# Replace range(...) with your eval_loader:
#   for batch in eval_loader:
#       x      = batch['data'].to(device)
#       target = batch['original'][..., :3].to(device)

DEMO_EVAL_STEPS = 8

with torch.no_grad():
    for _ in range(DEMO_EVAL_STEPS):
        # ── synthetic data ────────────────────────────────────────────────────
        x      = torch.randn(DEMO_BATCH, DEMO_T, arch["num_joints"], arch["dim_in"],  device=device)
        target = torch.randn(DEMO_BATCH, DEMO_T, arch["num_joints"], 3,               device=device)
        # ─────────────────────────────────────────────────────────────────────

        reconstructed, bundle = model(x)
        pred_xyz = reconstructed[..., :3]

        eval_losses.append(loss_mpjpe(pred_xyz, target).item())
        all_embeddings.append(bundle["pooled_pre_logits_embeddings"].cpu().numpy())

embeddings_array = np.concatenate(all_embeddings, axis=0)   # [N_samples, embed_dim]

print(f"Eval MPJPE (reconstruction) : {np.mean(eval_losses):.4f}")
print(f"Collected embeddings shape  : {embeddings_array.shape}")
print("\nEmbeddings are ready for downstream probing (e.g. ridge regression, logistic regression).")

Eval MPJPE (reconstruction) : 1.6034
Collected embeddings shape  : (16, 96)

Embeddings are ready for downstream probing (e.g. ridge regression, logistic regression).


## 8  Visualize: true vs predicted skeleton (all samples)

Recreates the README-style cartoon for every clip in `sample_data/` — CSV **and** NTDS —
overlaying the **Original** (input) skeleton against the model's **Predicted** reconstruction,
drawn in the **camera's own image plane (the front POV, no rotation)**.

Discovery is name-agnostic (walks folders by extension), so renaming or adding sessions needs
no code change. Each NTDS session has **two cameras**; the model was trained on the **front-facing
camera** (the subject faces it — shoulders spread across the camera's X axis), so we auto-select
that camera per session (`front_view_score`). The side/profile camera is off-distribution and
reconstructs ~4x worse (MPJPE ~0.037 vs ~0.008).

Section 6 perturbed the weights in-place, so we reload the pristine checkpoint first. One GIF
per sample is written to `Notebooks/reconstruction_gifs/` and shown inline.

> **Data note:** the NTDS clips are real subject-motion captures (folders use anonymized IDs).
> The GIF folder and the raw `.ntds` recordings are git-ignored — do not commit the recordings
> or the derived animations.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.animation import FuncAnimation, PillowWriter
from model.preprocessing.joints_file import noise_bones

# Section 6 mutated the weights in-place — restore the pristine checkpoint for a fair viz.
model.load_state_dict(state)
model = model.to(device).eval()

_BONES = list(noise_bones)            # 26-joint connectivity (after noise-joint removal)
_ORIG, _PRED = "#1f4fd8", "#1a9e3a"   # blue = Original (input), green = Predicted (reconstruction)

def animate_true_vs_pred(true_xyz, pred_xyz, out_path, title="", n_frames=72, fps=12):
    """Overlay Original vs Predicted skeletons in the camera image plane (front POV) → GIF.

    true_xyz / pred_xyz : [T, 26, 3] in the model's normalized coordinate frame.
    Horizontal axis = camera width (X), vertical = height (Y, drawn head-up) — the camera POV, no rotation.
    """
    ax_h, ax_v = 0, 1
    T = true_xyz.shape[0]
    idx = np.linspace(0, T - 1, min(n_frames, T)).astype(int)
    tr, pr = true_xyz[idx], pred_xyz[idx]
    proj = np.concatenate([tr, pr], 0)[..., [ax_h, ax_v]]
    lo, hi = proj.min((0, 1)), proj.max((0, 1))
    pad = 0.12 * (hi - lo + 1e-6)
    handles = [Line2D([], [], color=_ORIG, lw=2, label="Original"),
               Line2D([], [], color=_PRED, lw=2, label="Predicted")]

    fig, ax = plt.subplots(figsize=(6, 5.2))
    def draw(i):
        ax.clear()
        for pts, col in [(tr[i], _ORIG), (pr[i], _PRED)]:
            for a, b in _BONES:
                ax.plot([pts[a, ax_h], pts[b, ax_h]], [pts[a, ax_v], pts[b, ax_v]],
                        "-", color=col, lw=1.7, alpha=0.9)
            ax.scatter(pts[:, ax_h], pts[:, ax_v], s=9, color=col, zorder=3)
        ax.set_xlim(lo[0] - pad[0], hi[0] + pad[0])
        ax.set_ylim(hi[1] + pad[1], lo[1] - pad[1])          # invert Y so the head is up
        ax.set_aspect("equal"); ax.axis("off")
        ax.text(0.02, 0.02, f"Frame: {i + 1}/{len(idx)}", transform=ax.transAxes,
                va="bottom", ha="left", fontsize=9,
                bbox=dict(boxstyle="round", fc="white", ec="0.7"))
        ax.legend(handles=handles, loc="upper right", fontsize=9, frameon=False)
        ax.set_title(title, fontsize=10)

    FuncAnimation(fig, draw, frames=len(idx), interval=1000 / fps).save(
        out_path, writer=PillowWriter(fps=fps))
    plt.close(fig)
    return out_path

In [ ]:
from IPython.display import Image as IPyImage, display

GIF_DIR = os.path.join(project_root, "Notebooks", "reconstruction_gifs")
os.makedirs(GIF_DIR, exist_ok=True)

# raw k4abt joint indices (before noise-joint removal)
_SH_L, _SH_R, _HIP_L, _HIP_R = 5, 12, 18, 22

def front_view_score(path):
    """Higher = more front-on (facing the camera): the left/right body spread lies along the
    camera's X axis and is thin in depth (Z) — the model's training view. A side/profile camera
    shows the opposite (spread in Z). Used to pick the right camera per session.
    (The shipped CSV is front-on: shoulder x≫z.)"""
    raw = DualCameraDataset._get_np_from_df(_shim, load_skeleton_df(path))[:, :, :3]   # [T,32,3] mm
    pairs = [(_SH_L, _SH_R), (_HIP_L, _HIP_R)]
    dx = np.mean([np.abs(raw[:, a, 0] - raw[:, b, 0]).mean() for a, b in pairs])
    dz = np.mean([np.abs(raw[:, a, 2] - raw[:, b, 2]).mean() for a, b in pairs])
    return dx / (dz + 1e-6)

def discover_samples(root):
    """One clip per session folder. Prefer the CSV; for multi-camera NTDS sessions pick the
    front-facing camera (the model's training view; the side/profile camera is off-distribution
    and reconstructs ~4x worse: MPJPE ~0.037 vs ~0.008)."""
    found = []
    for dirpath, _, files in os.walk(root):
        if os.path.basename(dirpath).startswith("."):        # skip .ipynb_checkpoints etc.
            continue
        csvs = sorted(os.path.join(dirpath, f) for f in files if f.endswith(".csv"))
        ntds = sorted(os.path.join(dirpath, f) for f in files if f.endswith(".ntds"))
        if csvs:
            found.append(csvs[0])
        elif len(ntds) == 1:
            found.append(ntds[0])
        elif ntds:
            found.append(max(ntds, key=front_view_score))     # front-facing camera (the training view)
    return sorted(found)

SAMPLES     = discover_samples(os.path.join(project_root, "sample_data"))
SHOW_INLINE = True   # set False to only write GIFs to disk (no inline embedding)

print(f"Found {len(SAMPLES)} sample clip(s):\n")
for path in SAMPLES:
    rel = os.path.relpath(path, os.path.join(project_root, "sample_data"))
    x, n_raw = preprocess_sample(path)
    with torch.no_grad():
        recon, _ = model(x)
    mpjpe = loss_mpjpe(recon[..., :3], x[..., :3]).item()
    true_xyz = x[0, ..., :3].cpu().numpy()
    pred_xyz = recon[0, ..., :3].cpu().numpy()

    gif_name = rel.replace(os.sep, "__").rsplit(".", 1)[0] + ".gif"
    gif_path = os.path.join(GIF_DIR, gif_name)
    animate_true_vs_pred(true_xyz, pred_xyz, gif_path,
                         title=f"{os.path.basename(path)[:34]}  (MPJPE={mpjpe:.3f})")

    print(f"  {rel:58s} raw={n_raw:5d}  MPJPE={mpjpe:.4f}  →  {os.path.relpath(gif_path, project_root)}")
    if SHOW_INLINE:
        display(IPyImage(filename=gif_path))